<a href="https://colab.research.google.com/github/roalddalhwriter/isro/blob/main/HiRISE_VRC_Pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HiRISE Rootless Cone Detection — Pipeline v2
**Author:** Parth Gupta

### What this notebook implements
| Stage | Description |
|---|---|
| 1 | Install dependencies |
| 2 | Mount Drive + set all paths |
| 3 | Load DTM |
| 4 | Export browse JPG from full orthoimage |
| 5 | Build reference library from 14 verified Mars cone CSVs |
| 6 | SAHI + YOLOv8s candidate detection (conf ≥ 0.30) |
| 7 | Coordinate reprojection: browse px → ortho px → geo → DTM px |
| 8 | Confidence + size pre-filter (conf ≥ 0.50, width ≥ 150 m) |
| 9 | **Crater-floor snapping** — centroid to local DTM **minimum** (v2 fix) |
| 10 | **16-direction radial profile sampling** at 11.25° intervals |
| 11 | **Structural validation** — double-peak with centred dip (primary gate) |
| 12 | **Cosine similarity** vs reference library (secondary ranking) |
| 13 | Final validated cone set + morphometric extraction |
| 14 | Output plots + CSV |

In [1]:
!pip install rasterio sahi ultralytics scipy -q

import rasterio
import numpy as np
import os
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sahi.predict import get_sliced_prediction
from sahi.models.ultralytics import UltralyticsDetectionModel
from rasterio.transform import rowcol
from scipy.ndimage import map_coordinates
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
import pandas as pd

print('✅ Dependencies ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.0 MB/s eta 0:00:00
✅ Dependencies ready


## Cell 2 — Mount Drive & Set Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# ── Paths (from ElevationNewpipe) ─────────────────────────────────────────────
ORTHO_PATH    = '/content/drive/MyDrive/Elevation/PSP_007882_2065_RED_A_01_ORTHO.JP2'
DTM_PATH      = '/content/drive/MyDrive/Elevation/DTEEC_007671_2065_007882_2065_U01.IMG'
YOLO_WEIGHTS  = '/content/drive/MyDrive/Cones/runs/Icelandplusvers/weights/best.pt'
BROWSE_JPG    = '/content/drive/MyDrive/Elevation/ortho_browse.jpg'
REFERENCE_DIR = '/content/drive/MyDrive/Elevation New/mars'   # 14 verified cone CSVs
OUTPUT_DIR    = '/content/drive/MyDrive/cones/results_v5/'

SCALE_FACTOR  = 0.0515   # brings full ortho (~0.252 m/px) to ~4.9 m/px browse

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR + 'validated_cones/', exist_ok=True)
print('✅ Paths set')

Mounted at /content/drive
✅ Paths set


## Cell 3 — Load DTM

In [ ]:
with rasterio.open(DTM_PATH) as src:
    dtm_raw       = src.read(1).astype(np.float32)
    dtm_transform = src.transform
    dtm_nodata    = src.nodata
    scale         = src.scales[0]  if src.scales  else 1.0
    offset        = src.offsets[0] if src.offsets else 0.0

dtm = dtm_raw * scale + offset
if dtm_nodata is not None:
    dtm[dtm_raw == dtm_nodata] = np.nan

DTM_PX_M = abs(dtm_transform.a)
print(f'DTM shape     : {dtm.shape}')
print(f'DTM px scale  : {DTM_PX_M:.3f} m/px')
print(f'Elevation     : {np.nanmin(dtm):.1f} m → {np.nanmax(dtm):.1f} m')
print('✅ DTM loaded')

## Cell 4 — Export Browse JPG
Downsamples the full orthoimage to ~4.9 m/px to match training resolution.
p2/p98 percentile stretch handles the narrow 96–918 pixel value range in HiRISE data.

In [ ]:
with rasterio.open(ORTHO_PATH) as src:
    full_w          = src.width
    full_h          = src.height
    ortho_px_m      = abs(src.transform.a)
    ortho_transform = src.transform

    out_w = int(full_w * SCALE_FACTOR)
    out_h = int(full_h * SCALE_FACTOR)

    print(f'Full ortho   : {full_w} × {full_h} px at {ortho_px_m:.4f} m/px')
    print(f'Browse target: {out_w} × {out_h} px at {ortho_px_m / SCALE_FACTOR:.2f} m/px')

    band = src.read(1, out_shape=(out_h, out_w)).astype(np.float32)

# p2/p98 stretch — HiRISE values only span ~96-918 of 0-65535
p2, p98 = np.percentile(band[band > 0], (2, 98))
img_u8  = np.clip((band - p2) / (p98 - p2 + 1e-6) * 255, 0, 255).astype(np.uint8)

cv2.imwrite(BROWSE_JPG, img_u8)
print(f'✅ Browse JPG saved → {BROWSE_JPG}')

## Cell 5 — Build Reference Library
Loads the 14 manually verified Martian cone profiles extracted in QGIS.
Each is resampled to 100 points and normalised 0–1 to form shape templates.

In [ ]:
def normalize_profile(elev, n_points=100):
    """Resample to fixed length and normalize to 0-1."""
    elev  = np.array(elev, dtype=np.float32)
    x_old = np.linspace(0, 1, len(elev))
    x_new = np.linspace(0, 1, n_points)
    resampled = np.interp(x_new, x_old, elev)
    rng = resampled.max() - resampled.min()
    return (resampled - resampled.min()) / rng if rng > 0 else np.zeros(n_points)


reference_library = []
for f in sorted(Path(REFERENCE_DIR).glob('*.csv')):
    df = pd.read_csv(f)
    df.columns = [c.strip().lower() for c in df.columns]
    elev_col = [c for c in df.columns if 'elev' in c or 'z' in c][0]
    elev = df[elev_col].values
    template = normalize_profile(elev, n_points=100)
    reference_library.append({'name': f.stem, 'template': template})

print(f'✅ Reference library: {len(reference_library)} verified cone profiles')
for r in reference_library:
    print(f'   {r["name"]}')

## Cell 6 — SAHI + YOLOv8s Candidate Detection
Confidence threshold is deliberately loose (0.30) — the DEM pipeline does the real filtering.
YOLO is a circular-feature detector at this stage; cone vs crater discrimination happens in Cells 9–12.

In [ ]:
detection_model = UltralyticsDetectionModel(
    model_path           = YOLO_WEIGHTS,
    confidence_threshold = 0.30,   # loose — DEM validation is the primary filter
    device               = 'cuda:0'
)

result = get_sliced_prediction(
    BROWSE_JPG,
    detection_model,
    slice_height         = 1260,   # matches training tile dimensions
    slice_width          = 1260,
    overlap_height_ratio = 0.2,
    overlap_width_ratio  = 0.2,
    verbose              = 1
)

print(f'✅ Raw SAHI detections: {len(result.object_prediction_list)}')

## Cell 7 — Coordinate Reprojection
Three-step chain:
1. Browse px → full ortho px (÷ SCALE_FACTOR)
2. Full ortho px → geographic coords (ortho affine transform)
3. Geographic coords → DTM px (DTM affine transform)

Both files share the Mars Equirectangular CRS — no CRS conversion needed, only affine transforms.

In [ ]:
candidates = []
for op in result.object_prediction_list:
    xmin, ymin, xmax, ymax = op.bbox.to_xyxy()
    conf = op.score.value

    # Centroid + size in browse JPG pixel space
    cx_browse    = (xmin + xmax) / 2.0
    cy_browse    = (ymin + ymax) / 2.0
    bbox_w_browse = xmax - xmin

    # Step 1 — browse px → full ortho px
    cx_ortho = cx_browse / SCALE_FACTOR
    cy_ortho = cy_browse / SCALE_FACTOR

    # Step 2 — full ortho px → geographic coords (Mars Equirectangular)
    geo_x, geo_y = ortho_transform * (cx_ortho, cy_ortho)

    # Step 3 — geographic coords → DTM px
    dtm_r, dtm_c = rowcol(dtm_transform, geo_x, geo_y)

    # Radius in DTM pixels
    radius_m   = (bbox_w_browse / 2.0) * (ortho_px_m / SCALE_FACTOR)
    radius_dtm = max(int(radius_m / DTM_PX_M), 15)
    bbox_w_m   = bbox_w_browse * (ortho_px_m / SCALE_FACTOR)

    candidates.append({
        'cx_browse' : cx_browse,
        'cy_browse' : cy_browse,
        'cx_dtm'    : int(dtm_c),
        'cy_dtm'    : int(dtm_r),
        'radius_dtm': radius_dtm,
        'bbox_w_m'  : bbox_w_m,
        'conf'      : conf,
    })

print(f'✅ {len(candidates)} candidates reprojected to DTM space')

## Cell 8 — Confidence + Size Pre-filter
Drop clear noise before running the expensive DEM profiling:
- conf ≥ 0.50 (tighter threshold post-reprojection)
- bbox width ≥ 150 m (consistent with known VRC dimensions in Acidalia Planitia)

In [ ]:
MIN_CONF   = 0.50
MIN_SIZE_M = 150.0

candidates = [
    c for c in candidates
    if c['conf'] >= MIN_CONF and c['bbox_w_m'] >= MIN_SIZE_M
]

sizes = [c['bbox_w_m'] for c in candidates]
print(f'✅ After conf ≥ {MIN_CONF} + size ≥ {MIN_SIZE_M} m: {len(candidates)} candidates')
if sizes:
    print(f'   Bbox width — min: {np.min(sizes):.0f} m  '
          f'max: {np.max(sizes):.0f} m  '
          f'mean: {np.mean(sizes):.0f} m')

## Cell 9 — Crater-Floor Snapping ✅ (v2 fix)
**v1 bug:** centroid was snapped to `nanargmax` (rim peak) → profile centre sits on the summit
→ crater depth measured as summit − summit ≈ 0 for every candidate.

**v2 fix:** snap to `nanargmin` (crater floor) within the search radius.
The profile centre now sits in the depression, so both rim peaks appear on either side.

In [ ]:
def snap_to_crater_floor(dtm, cx, cy, search_radius):
    """
    Snap centroid to the local DTM MINIMUM (crater floor) within search_radius.
    This is the critical v2 fix — v1 used nanargmax (rim peak) which caused
    crater depth to read near zero for all candidates.
    """
    r0 = max(0, cy - search_radius)
    r1 = min(dtm.shape[0], cy + search_radius)
    c0 = max(0, cx - search_radius)
    c1 = min(dtm.shape[1], cx + search_radius)

    chip = dtm[r0:r1, c0:c1]
    if np.all(np.isnan(chip)):
        return cx, cy  # can't snap — return original

    floor_idx         = np.nanargmin(chip)
    floor_r, floor_c  = np.unravel_index(floor_idx, chip.shape)
    return int(c0 + floor_c), int(r0 + floor_r)


for c in candidates:
    new_cx, new_cy = snap_to_crater_floor(dtm, c['cx_dtm'], c['cy_dtm'], c['radius_dtm'])
    c['cx_dtm'] = new_cx
    c['cy_dtm'] = new_cy

print('✅ All centroids snapped to local DTM minimum (crater floor)')

## Cell 10 — 16-Direction Radial Profile Sampling ✅ (v2 fix)
**v1 bug:** fixed E-W and N-S only — misses the true cross-section for cones
whose elongation axis is not cardinal.

**v2 fix:** 16 radial profiles per candidate at 11.25° intervals (0°, 11.25°, ..., 168.75°).
At least one direction will pass near-optimally through the cone's true diameter
regardless of orientation. Each profile extends 1.5× the detected cone radius
on both sides of the crater centre, sampled at 100 points with bilinear interpolation.

In [ ]:
ANGLES    = [i * 11.25 for i in range(16)]   # 0° to 168.75°
N_SAMPLES = 100


def extract_radial_profile(dtm, cx, cy, length_px, angle_deg, n_samples=N_SAMPLES):
    """
    Sample elevation along a radial line centred on (cx, cy).
    Extends length_px in both directions (total span = 2 × length_px).
    Uses bilinear interpolation via scipy.ndimage.map_coordinates.
    """
    angle_rad = np.radians(angle_deg)
    dx = np.cos(angle_rad)
    dy = np.sin(angle_rad)

    t    = np.linspace(-length_px, length_px, n_samples)
    cols = np.clip(cx + t * dx, 0, dtm.shape[1] - 1)
    rows = np.clip(cy + t * dy, 0, dtm.shape[0] - 1)

    profile = map_coordinates(dtm, [rows, cols], order=1, mode='nearest')
    profile[profile < -9000] = np.nan
    return profile


for c in candidates:
    length_px = c['radius_dtm'] * 1.5
    c['radial_profiles'] = [
        extract_radial_profile(dtm, c['cx_dtm'], c['cy_dtm'], length_px, angle)
        for angle in ANGLES
    ]

print(f'✅ 16-direction radial profiles extracted for {len(candidates)} candidates')
print(f'   Profile length: 1.5 × radius (each direction), {N_SAMPLES} samples, bilinear interp')

## Cell 11 — Structural Validation (Primary Gate) ✅
**Why this is the primary gate, not cosine similarity:**
In testing, cosine similarity alone at a threshold of 0.99 retained 92% false positives
(165 of 180 candidates that subsequently failed the structural check). Shape-template matching
cannot distinguish a rootless cone from a crater because one side of a crater profile
still achieves high cosine similarity against the cone template.

**Structural check — all three conditions must hold:**
1. At least 2 peaks with prominence ≥ 3 m (`scipy.signal.find_peaks`)
2. A dip between the two most prominent peaks meaningfully lower than both
3. The dip is roughly centred — within 35% of peak separation from the midpoint

In [ ]:
MIN_PEAK_PROMINENCE = 3.0   # metres
MAX_DIP_OFFSET      = 0.35  # dip must be within 35% of peak separation from midpoint


def validate_cone_structure(profile, min_prominence=MIN_PEAK_PROMINENCE):
    """
    Primary gate: check for double-peak with centred dip.
    Returns (is_valid: bool, info: dict).

    A rootless cone cross-section has the form:
        baseline → left rim peak → crater depression → right rim peak → baseline
    This enforces that specific topographic logic rather than just rewarding
    any profile that vaguely resembles the reference waveform.
    """
    # Fill NaNs by linear interpolation before peak detection
    valid = ~np.isnan(profile)
    if valid.sum() < len(profile) * 0.5:
        return False, {'reason': 'too many NaNs'}

    p = profile.copy()
    if np.any(~valid):
        p = np.interp(np.arange(len(p)), np.where(valid)[0], p[valid])

    # Find all peaks with minimum prominence
    peaks, props = find_peaks(p, prominence=min_prominence)

    if len(peaks) < 2:
        return False, {'reason': f'only {len(peaks)} peak(s) found (need ≥ 2)'}

    # Select the two most prominent peaks
    prominences  = props['prominences']
    top2_idx     = np.argsort(prominences)[-2:]
    peak_a, peak_b = sorted(peaks[top2_idx])   # left peak, right peak

    # Find minimum (dip) between the two peaks
    between      = p[peak_a:peak_b + 1]
    dip_rel_idx  = int(np.argmin(between))
    dip_abs_idx  = peak_a + dip_rel_idx
    dip_val      = p[dip_abs_idx]

    # Condition 1: dip must be meaningfully lower than both peaks
    elev_a, elev_b = p[peak_a], p[peak_b]
    lower_peak     = min(elev_a, elev_b)
    if dip_val >= lower_peak:
        return False, {'reason': 'dip not lower than both peaks'}

    # Condition 2: dip must be roughly centred between the two peaks
    # (within 35% of peak separation from the geometric midpoint)
    midpoint      = (peak_a + peak_b) / 2.0
    peak_sep      = peak_b - peak_a
    dip_offset    = abs(dip_abs_idx - midpoint)
    if dip_offset > MAX_DIP_OFFSET * peak_sep:
        return False, {'reason': f'dip off-centre (offset={dip_offset/peak_sep:.2f} of sep)'}

    return True, {
        'peak_a_idx'   : int(peak_a),
        'peak_b_idx'   : int(peak_b),
        'dip_idx'      : int(dip_abs_idx),
        'elev_peak_a'  : round(float(elev_a), 2),
        'elev_peak_b'  : round(float(elev_b), 2),
        'elev_dip'     : round(float(dip_val), 2),
        'dip_offset_frac': round(float(dip_offset / peak_sep), 3),
    }


print('✅ validate_cone_structure() defined')
print(f'   min_prominence = {MIN_PEAK_PROMINENCE} m')
print(f'   max_dip_offset = {MAX_DIP_OFFSET} (fraction of peak separation)')

## Cell 12 — Cosine Similarity vs Reference Library (Secondary Ranking)
For each candidate, across all 16 radial profiles:
1. Run structural validation on every direction
2. Compute cosine similarity against all 14 reference templates
3. **Among structurally valid directions**, select the one with highest cosine similarity
4. If **no** direction passes structural validation, take the best cosine match anyway but flag it as `used_structural_fallback=True`

The final `similarity_score` is from the selected direction's best reference match.

In [ ]:
def best_direction_and_score(radial_profiles, reference_library):
    """
    Two-stage classification across all 16 radial directions:

    Stage 1 (primary gate)  — structural validation (double-peak + centred dip)
    Stage 2 (secondary rank) — cosine similarity vs 14 reference templates

    Returns a dict with the winning direction's info, or None if all profiles
    have too many NaNs to evaluate.
    """
    dir_results = []

    for angle_idx, prof in enumerate(radial_profiles):
        # Skip profiles with too many missing values
        nan_frac = np.isnan(prof).mean()
        if nan_frac > 0.20:
            continue
        valid = ~np.isnan(prof)
        if valid.sum() < len(prof) * 0.5:
            continue

        # NaN-fill for processing
        p = np.interp(np.arange(len(prof)), np.where(valid)[0], prof[valid])

        # Stage 1 — structural validation
        struct_ok, struct_info = validate_cone_structure(p)

        # Stage 2 — cosine similarity vs all reference templates
        norm_p   = normalize_profile(p, n_points=100)
        best_sim = -1.0
        best_ref = None
        for ref in reference_library:
            sim = 1.0 - cosine(norm_p, ref['template'])
            if sim > best_sim:
                best_sim = sim
                best_ref = ref['name']

        dir_results.append({
            'angle_idx'  : angle_idx,
            'angle_deg'  : ANGLES[angle_idx],
            'struct_ok'  : struct_ok,
            'struct_info': struct_info,
            'similarity' : best_sim,
            'best_ref'   : best_ref,
            'profile_filled': p,
        })

    if not dir_results:
        return None

    # Prefer structurally valid directions; among those, pick highest similarity
    valid_dirs = [d for d in dir_results if d['struct_ok']]
    if valid_dirs:
        winner = max(valid_dirs, key=lambda d: d['similarity'])
        winner['used_structural_fallback'] = False
    else:
        # No direction passed structural validation — take best cosine match anyway
        winner = max(dir_results, key=lambda d: d['similarity'])
        winner['used_structural_fallback'] = True

    return winner


# ── Run on all candidates ─────────────────────────────────────────────────────
for c in candidates:
    result_dir = best_direction_and_score(c['radial_profiles'], reference_library)

    if result_dir is None:
        c['similarity_score']          = -1.0
        c['struct_ok']                  = False
        c['best_angle_idx']             = None
        c['best_angle_deg']             = None
        c['best_ref']                   = None
        c['used_structural_fallback']   = True
        c['struct_info']                = {}
    else:
        c['similarity_score']          = result_dir['similarity']
        c['struct_ok']                  = result_dir['struct_ok']
        c['best_angle_idx']             = result_dir['angle_idx']
        c['best_angle_deg']             = result_dir['angle_deg']
        c['best_ref']                   = result_dir['best_ref']
        c['used_structural_fallback']   = result_dir['used_structural_fallback']
        c['struct_info']                = result_dir['struct_info']

# ── Summary ───────────────────────────────────────────────────────────────────
n_total      = len(candidates)
n_struct     = sum(1 for c in candidates if c['struct_ok'])
n_fallback   = sum(1 for c in candidates if c['used_structural_fallback'])
scores_valid = [c['similarity_score'] for c in candidates if c['struct_ok']]

print(f'Candidates evaluated          : {n_total}')
print(f'Passed structural validation  : {n_struct}')
print(f'Used cosine-only fallback     : {n_fallback}')
if scores_valid:
    print(f'\nAmong structurally valid candidates:')
    print(f'  Min similarity  : {np.min(scores_valid):.3f}')
    print(f'  Max similarity  : {np.max(scores_valid):.3f}')
    print(f'  Mean similarity : {np.mean(scores_valid):.3f}')
print('\n✅ Two-stage classification complete')

## Cell 13 — Extract Morphometrics & Produce Final Validated Set
Compute cone height, crater depth, width, and H/W ratio from the winning
direction's profile for each structurally validated candidate.

In [ ]:
def extract_morphometrics(profile_filled, radius_dtm):
    """
    Extract morphometric measurements from the NaN-filled winning profile.
    Measurements are in metres (using DTM_PX_M for distance conversion).
    """
    n       = len(profile_filled)
    p       = profile_filled

    # Base: mean of first and last 5 samples (profile endpoints)
    base_elev   = float(np.nanmean(np.concatenate([p[:5], p[-5:]])))
    summit_elev = float(np.nanmax(p))
    center_elev = float(p[n // 2])      # crater floor — profile is centred on DTM minimum
    cone_height  = summit_elev - base_elev
    crater_depth = summit_elev - center_elev
    width_m      = 2.0 * radius_dtm * 1.5 * DTM_PX_M   # full profile span in metres
    h_w_ratio    = cone_height / width_m if width_m > 0 else 0.0

    return {
        'base_elev_m'   : round(base_elev,   2),
        'summit_elev_m' : round(summit_elev, 2),
        'center_elev_m' : round(center_elev, 2),
        'cone_height_m' : round(cone_height, 2),
        'crater_depth_m': round(crater_depth, 2),
        'width_m'       : round(width_m,      2),
        'h_w_ratio'     : round(h_w_ratio,    4),
    }


# ── Apply morphometric extraction to structurally validated candidates ────────
validated = []
for c in candidates:
    if not c['struct_ok']:
        continue

    angle_idx = c['best_angle_idx']
    prof_raw  = c['radial_profiles'][angle_idx]
    valid     = ~np.isnan(prof_raw)
    p_filled  = np.interp(np.arange(len(prof_raw)),
                          np.where(valid)[0], prof_raw[valid])

    metrics = extract_morphometrics(p_filled, c['radius_dtm'])
    c.update(metrics)
    validated.append(c)

print(f'✅ Final validated cones: {len(validated)}')

if validated:
    print(f'\nMorphometric summary:')
    for key in ['cone_height_m', 'crater_depth_m', 'width_m', 'h_w_ratio']:
        vals = [c[key] for c in validated]
        print(f'  {key:<20} mean={np.mean(vals):.2f}  '
              f'std=±{np.std(vals):.2f}  '
              f'min={np.min(vals):.2f}  max={np.max(vals):.2f}')

## Cell 14 — Per-Cone Output Plots
For each validated cone: DTM chip with profile direction overlaid, raw profile, and
normalised profile vs best-matching reference template.

In [ ]:
for i, c in enumerate(validated):
    angle_idx = c['best_angle_idx']
    angle_deg = c['best_angle_deg']
    prof_raw  = c['radial_profiles'][angle_idx]

    valid    = ~np.isnan(prof_raw)
    p_filled = np.interp(np.arange(len(prof_raw)),
                         np.where(valid)[0], prof_raw[valid])
    norm_p   = normalize_profile(p_filled, n_points=100)
    ref_tmpl = [r['template'] for r in reference_library
                if r['name'] == c['best_ref']][0]

    dist_m = np.linspace(
        -c['radius_dtm'] * 1.5 * DTM_PX_M,
         c['radius_dtm'] * 1.5 * DTM_PX_M,
        len(p_filled)
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # ── Panel 1: DTM chip with profile line ──────────────────────────
    half = c['radius_dtm'] * 3
    r0 = max(0, c['cy_dtm'] - half);  r1 = min(dtm.shape[0], c['cy_dtm'] + half)
    c0 = max(0, c['cx_dtm'] - half);  c1 = min(dtm.shape[1], c['cx_dtm'] + half)
    chip = dtm[r0:r1, c0:c1]

    axes[0].imshow(chip, cmap='terrain',
                   vmin=np.nanpercentile(chip, 2),
                   vmax=np.nanpercentile(chip, 98))
    axes[0].plot(c['cx_dtm'] - c0, c['cy_dtm'] - r0,
                 'r+', markersize=12, mew=2, label='crater floor')

    length_px = c['radius_dtm'] * 1.5
    angle_rad = np.radians(angle_deg)
    dx, dy = np.cos(angle_rad), np.sin(angle_rad)
    x1 = c['cx_dtm'] - c0 - length_px * dx
    y1 = c['cy_dtm'] - r0 - length_px * dy
    x2 = c['cx_dtm'] - c0 + length_px * dx
    y2 = c['cy_dtm'] - r0 + length_px * dy
    axes[0].plot([x1, x2], [y1, y2], color='cyan', lw=1.5, alpha=0.9)
    axes[0].set_title(
        f'Cone {i}  conf={c["conf"]:.2f}  {angle_deg}°', fontsize=9)
    axes[0].axis('off')

    # ── Panel 2: Raw elevation profile ───────────────────────────────
    axes[1].plot(dist_m, p_filled, color='#c0392b', lw=1.5)
    axes[1].axvline(0, color='grey', lw=0.8, linestyle='--', alpha=0.6)
    axes[1].set_xlabel('Distance from crater centre (m)')
    axes[1].set_ylabel('Elevation (m)')
    axes[1].set_title(
        f'H={c["cone_height_m"]:.1f} m  '
        f'D={c["crater_depth_m"]:.1f} m  '
        f'W={c["width_m"]:.0f} m  '
        f'H/W={c["h_w_ratio"]:.3f}', fontsize=9)
    axes[1].grid(True, alpha=0.3)

    # ── Panel 3: Normalised candidate vs reference template ───────────
    axes[2].plot(norm_p,   color='#c0392b', lw=1.5, label='Candidate (norm.)')
    axes[2].plot(ref_tmpl, color='#2980b9', lw=1.5,
                 label=f'Ref: {c["best_ref"]}')
    axes[2].set_xlabel('Profile sample index')
    axes[2].set_ylabel('Normalised elevation')
    axes[2].set_title(f'Cosine similarity = {c["similarity_score"]:.3f}', fontsize=9)
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(
        f'Validated Cone {i}  |  size={c["bbox_w_m"]:.0f} m  '
        f'|  conf={c["conf"]:.2f}  |  ref={c["best_ref"]}',
        fontsize=10)
    plt.tight_layout()
    out_path = OUTPUT_DIR + f'validated_cones/cone_{i:03d}_sim{c["similarity_score"]:.3f}.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'[{i+1}/{len(validated)}] saved → {out_path}')

print(f'\n✅ All {len(validated)} cone plots saved to {OUTPUT_DIR}validated_cones/')

## Cell 15 — Browse-Image Detection Map

In [ ]:
from PIL import Image as PILImage

img      = PILImage.open(BROWSE_JPG).convert('RGB')
img_arr  = np.array(img)

fig, ax = plt.subplots(figsize=(10, 16), dpi=130)
ax.imshow(img_arr)

for c in validated:
    cx = c['cx_browse']
    cy = c['cy_browse']
    w  = c['bbox_w_m'] * SCALE_FACTOR / ortho_px_m   # back to browse pixels

    color = 'lime' if c['conf'] >= 0.70 else ('yellow' if c['conf'] >= 0.50 else 'red')

    rect = patches.Rectangle(
        (cx - w / 2, cy - w / 2), w, w,
        linewidth=0.9, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)

    # Profile direction line
    length_browse = (c['radius_dtm'] * 1.5 * DTM_PX_M) / (ortho_px_m / SCALE_FACTOR)
    angle_rad = np.radians(c['best_angle_deg'])
    dx, dy = np.cos(angle_rad), np.sin(angle_rad)
    ax.plot([cx - length_browse * dx, cx + length_browse * dx],
            [cy - length_browse * dy, cy + length_browse * dy],
            color='cyan', lw=0.7, alpha=0.8)

ax.plot([], [], 's', color='lime',   label='conf ≥ 0.70', markersize=6)
ax.plot([], [], 's', color='yellow', label='conf ≥ 0.50', markersize=6)
ax.plot([], [], 's', color='red',    label='conf < 0.50',  markersize=6)
ax.plot([], [], '-', color='cyan',   label='best profile direction')
ax.legend(loc='upper right', fontsize=8)
ax.set_title(f'Validated detections: {len(validated)} | v2 pipeline', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'detections_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Detection map saved')

## Cell 16 — Save Results CSV + Print Summary

In [ ]:
# ── Build results dataframe ───────────────────────────────────────────────────
rows = []
for i, c in enumerate(validated):
    rows.append({
        'cone_id'           : i,
        'cx_dtm'            : c['cx_dtm'],
        'cy_dtm'            : c['cy_dtm'],
        'bbox_w_m'          : round(c['bbox_w_m'],   1),
        'conf'              : round(c['conf'],        3),
        'best_angle_deg'    : c['best_angle_deg'],
        'similarity_score'  : round(c['similarity_score'], 4),
        'best_ref_match'    : c['best_ref'],
        'cone_height_m'     : c['cone_height_m'],
        'crater_depth_m'    : c['crater_depth_m'],
        'width_m'           : c['width_m'],
        'h_w_ratio'         : c['h_w_ratio'],
        'base_elev_m'       : c['base_elev_m'],
        'summit_elev_m'     : c['summit_elev_m'],
        'center_elev_m'     : c['center_elev_m'],
    })

df_out = pd.DataFrame(rows)
csv_path = OUTPUT_DIR + 'validated_cones_v2.csv'
df_out.to_csv(csv_path, index=False)
print(f'✅ Results saved → {csv_path}')
print()
print(df_out.to_string(index=False))

# ── Filtering funnel summary ──────────────────────────────────────────────────
print('\n' + '='*55)
print('DETECTION FILTERING FUNNEL')
print('='*55)
print(f'  Raw SAHI detections (conf ≥ 0.30)  : {len(result.object_prediction_list)}')
print(f'  After conf ≥ 0.50 + size ≥ 150 m   : {len([c for c in candidates])}')
print(f'  After structural + cosine validation : {len(validated)}')
print('='*55)

# ── Morphometric summary ──────────────────────────────────────────────────────
if validated:
    print('\nMORPHOMETRIC SUMMARY (validated cones)')
    print(f'  {"Metric":<22} {"Mean":>8}  {"±Std":>8}  {"Min":>8}  {"Max":>8}')
    print('  ' + '-'*54)
    for key, label in [
        ('cone_height_m',  'Cone Height (m)'),
        ('crater_depth_m', 'Crater Depth (m)'),
        ('width_m',        'Width (m)'),
        ('h_w_ratio',      'H/W Ratio'),
        ('similarity_score','Cosine Similarity'),
    ]:
        vals = [c[key] for c in validated]
        print(f'  {label:<22} {np.mean(vals):>8.3f}  '
              f'±{np.std(vals):>6.3f}  '
              f'{np.min(vals):>8.3f}  {np.max(vals):>8.3f}')

## Cell 17 — Similarity Diagnostic Plot

In [ ]:
# Show 5 candidates spanning the similarity score range
sorted_v = sorted(validated, key=lambda c: c['similarity_score'])
n_show   = min(5, len(sorted_v))
idxs     = [int(i * (len(sorted_v) - 1) / (n_show - 1)) for i in range(n_show)] if n_show > 1 else [0]

fig, axes = plt.subplots(n_show, 2, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = [axes]

for row, idx in enumerate(idxs):
    c         = sorted_v[idx]
    angle_idx = c['best_angle_idx']
    prof_raw  = c['radial_profiles'][angle_idx]
    valid     = ~np.isnan(prof_raw)
    p_filled  = np.interp(np.arange(len(prof_raw)),
                          np.where(valid)[0], prof_raw[valid])
    norm_p    = normalize_profile(p_filled, n_points=100)
    ref_tmpl  = [r['template'] for r in reference_library
                 if r['name'] == c['best_ref']][0]

    axes[row][0].plot(p_filled, color='#c0392b', lw=1.4)
    axes[row][0].set_title(
        f'Cone (sim={c["similarity_score"]:.3f}) — raw profile  '
        f'H={c["cone_height_m"]:.1f}m  D={c["crater_depth_m"]:.1f}m',
        fontsize=9)
    axes[row][0].grid(True, alpha=0.3)

    axes[row][1].plot(norm_p,   color='#c0392b', lw=1.4, label='Candidate (norm.)')
    axes[row][1].plot(ref_tmpl, color='#2980b9', lw=1.4, label=f'Ref: {c["best_ref"]}')
    axes[row][1].set_title(
        f'Normalised comparison  sim={c["similarity_score"]:.3f}', fontsize=9)
    axes[row][1].legend(fontsize=8)
    axes[row][1].grid(True, alpha=0.3)

plt.suptitle('Similarity Diagnostic — low → high score', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'similarity_diagnostic.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Similarity diagnostic saved')